# YOLO12-Small + GhostConv + EMA-32 with pretrained Ghost initialization

Varian ini memakai EMA-32 pada P3/8 dan GhostConv pada downsampling P4/16 serta P5/32. Berbeda dari notebook sebelumnya, GhostConv tidak lagi sepenuhnya random: bagian primary convolution dan BatchNorm diinisialisasi dari Conv yolo12s.pt yang digantikan. Cheap depthwise branch diinisialisasi sebagai identity non-random.

Ini adalah strategi inisialisasi untuk memperbaiki konvergensi dan berpotensi meningkatkan mAP. Jalankan baseline original YOLO12s dengan split, seed, hyperparameter, dan batch yang sama untuk membuktikan peningkatan.

> **Gunakan notebook `kaggle_yolo12s_ghost_ema32_pretrained_verified_rdd2022.ipynb` untuk eksperimen baru.** Notebook tersebut memverifikasi parameter trainable, memastikan trainer memakai objek hasil transfer, dan mencetak gradien aktual sebelum optimizer pertama. Notebook ini dipertahankan hanya untuk riwayat eksperimen.

In [ ]:
# 1. Clone source branch ini dan install implementasi custom. Aktifkan GPU dan Internet di Kaggle.
import json
import platform
import re
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ghost-ema-pretrained'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))

import torch
import ultralytics
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Commit      : {REPO_COMMIT}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Dataset, setting eksperimen yang sama dengan baseline, dan pemeriksaan arsitektur.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ghost-ema32.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML,
)
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED, EMA_FACTOR = 0, 2, 42, 32
EXPERIMENT_NAME = 'yolo12s_ghost_ema32_pretrained_init_ch_rdd2022'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists() and all(path.exists() for path in CUSTOM_SOURCE_FILES)

from ultralytics.nn.modules import EMAAttention, GhostConv
from ultralytics.nn.tasks import DetectionModel

log_section('YOLO12S + GHOSTCONV + EMA-32 MODEL INFO')
check_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=True)
ema_layers = [(layer.i, layer.groups) for layer in check_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
assert ema_layers == [(5, EMA_FACTOR)] and ghost_layers == [6, 8]
check_model.eval()
with torch.inference_mode():
    model_output = check_model(torch.zeros(1, 3, IMGSZ, IMGSZ))
assert isinstance(model_output, tuple) and model_output[0].shape == (1, 9, 8400)
print(f'Parameters (5 classes): {sum(p.numel() for p in check_model.parameters()):,}')
print('EMA layer             :', ema_layers)
print('GhostConv layers      :', ghost_layers)
check_model.info(detailed=False, verbose=True)
del check_model, model_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 3. Transfer yolo12s.pt, termasuk inisialisasi pretrained-aware untuk dua GhostConv.
from ultralytics import YOLO
from ultralytics.nn.modules import Conv

PRETRAINED_WEIGHTS = 'yolo12s.pt'
GHOST_SOURCE_TARGET_PAIRS = ((5, 6), (7, 8))

def target_layer_index(source_index: int) -> int:
    return source_index + int(source_index >= 5)  # EMA disisipkan setelah layer sumber 4.

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

def copy_bn_prefix(source_bn, target_bn, count: int) -> None:
    for name in ('weight', 'bias', 'running_mean', 'running_var', 'num_batches_tracked'):
        source_value, target_value = getattr(source_bn, name), getattr(target_bn, name)
        target_value.copy_(source_value if source_value.ndim == 0 else source_value[:count])

def initialize_ghost_from_pretrained_conv(source_conv: Conv, target_ghost: GhostConv) -> dict:
    primary_channels = target_ghost.cv1.conv.out_channels
    if source_conv.conv.weight.shape[1:] != target_ghost.cv1.conv.weight.shape[1:]:
        raise ValueError('Source Conv and GhostConv primary branch are incompatible.')
    with torch.no_grad():
        # Preserve half of the original pretrained Conv filters and their BatchNorm statistics.
        target_ghost.cv1.conv.weight.copy_(source_conv.conv.weight[:primary_channels])
        copy_bn_prefix(source_conv.bn, target_ghost.cv1.bn, primary_channels)
        # The cheap branch has no exact Conv equivalent; initialize a depthwise 5x5 identity instead of random weights.
        target_ghost.cv2.conv.weight.zero_()
        center_h, center_w = (size // 2 for size in target_ghost.cv2.conv.weight.shape[-2:])
        target_ghost.cv2.conv.weight[:, 0, center_h, center_w] = 1.0
        target_ghost.cv2.bn.weight.fill_(1.0)
        target_ghost.cv2.bn.bias.zero_()
        target_ghost.cv2.bn.running_mean.zero_()
        target_ghost.cv2.bn.running_var.fill_(1.0)
        target_ghost.cv2.bn.num_batches_tracked.zero_()
    return {'primary_channels_copied': primary_channels, 'cheap_branch': 'depthwise identity'}

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
GHOST_INITIALIZATION = []
for source_index, target_index in GHOST_SOURCE_TARGET_PAIRS:
    source_layer, target_layer = source_model.model[source_index], model.model.model[target_index]
    if not isinstance(source_layer, Conv) or not isinstance(target_layer, GhostConv):
        raise TypeError(f'Expected Conv->{GhostConv} at source {source_index}, target {target_index}.')
    GHOST_INITIALIZATION.append({
        'source_layer': source_index, 'target_layer': target_index,
        **initialize_ghost_from_pretrained_conv(source_layer, target_layer),
    })
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'standard_transferred_tensors': len(transferred_state), 'target_tensors': len(target_state),
    'uninitialized_before_ghost_init': len(incompatible.missing_keys),
    'ema_is_random': True, 'ghost_initialization': GHOST_INITIALIZATION,
}
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(json.dumps(PRETRAINED_REPORT, indent=2))
print('All compatible tensors are pretrained; Ghost primary branches now inherit Conv pretrained filters.')


In [ ]:
# 4. Training dengan setting identik baseline agar perbandingan mAP valid.
log_section('TRAINING STARTED - GHOSTCONV + EMA-32 PRETRAINED INIT')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
    seed=SEED, plots=True, verbose=True,
)
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 5. Evaluasi best.pt, simpan metrik, dan buat ZIP hasil.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
            'save_dir': str(metrics.save_dir)}

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                    conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                    name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML), 'ema_factor': EMA_FACTOR,
    'ghost_conv_positions': 'P4/16 and P5/32', 'pretrained_transfer': PRETRAINED_REPORT,
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER,
    'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
